# Party Movement, Crisis Response and Positions — zero-shot topics

This notebook covers:

1. **Party-level analysis**: what the party-year embeddings look like before any topic filtering.
2. **Analyze trajectories**: do the political events show up as movement
   in the embedding space? / Detect movement of parties triggered by crisis
3. **Party anchors**: where an axis can be
   anchored, where do the parties sit?
4. **Topic frequencies**: does an event change how much is said about its topics at all?

Topic assignments here come from the **zero-shot categories** of `05b_topic__analysis_zeroshot`
(`speech_topic_assignments_zeroshot.csv`), not from the BERTopic clusters of `05`. The categories are
coarse — 21 committee-style areas covering every speech — so each event rests on a much broader slice of
the corpus than in the BERTopic twin, and the two notebooks are not expected to agree.


## 0. Setup and data

In [ ]:
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from aggregation_functions import aggregate_by_mp_year, aggregate_by_party_year

In [ ]:
TOPICS     = "speech_topic_assignments_zeroshot.csv"          # speech -> topic, from notebook 05b
SPEECHES   = "jina_v3_contrastive_full.parquet"   # one row per speech, fine-tuned encoder
PARTY_YEAR = "party_year_embeddings_finetuned.parquet"   # party-year aggregates, from notebook 03

# one colour map for the whole notebook
PARTY_COLORS = {"CDU/CSU": "black", "SPD": "red", "Grüne": "green", "FDP": "gold",
                "LINKE": "purple", "AfD": "royalblue", "PDS": "darkorchid"}
party_colors = PARTY_COLORS   # parts 1 and 2 refer to it in lowercase

# figure styling, set once so every figure in the notebook looks the same
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# data frame with speech to topic assignments
df_topics = pd.read_csv(TOPICS)
df_topics

In [ ]:
# data frame with fine tuned embeddings
df_embeddings = pd.read_parquet(SPEECHES)
df_embeddings

In [ ]:
# merge embedding and topics data frame
df_embeddings_topics = df_embeddings.merge(
    df_topics.drop(columns=["party", "date"]),
    left_on="id", right_on="speech_id",
    how="inner"
)

df_embeddings_topics = df_embeddings_topics[df_embeddings_topics.party != 'Fraktionslos']
df_embeddings_topics

The anchored-position and frequency analysis in parts 3 and 4 works on the same merged table, only in a
slightly different shape: the placeholder speaker id `-1` is dropped and the embedding is kept as
a float32 array in the column `E`.

In [ ]:
df = df_embeddings_topics[df_embeddings_topics.politicianId != -1].copy()
df["E"] = df.embedding.map(lambda v: np.asarray(v, dtype="float32"))

ALL_TOPICS = sorted(df.topic_emb_full_name.unique())
print(f"{len(df):,} speeches | {len(ALL_TOPICS)} topics | years {df.year.min()}-{df.year.max()}")
print(df.party.value_counts().to_string())

## 1. Party-level analysis

Some graphs built from the party-year embeddings, before any topic filtering.

In [ ]:
#import file
py = pd.read_parquet(PARTY_YEAR)

#PCA analysis
X = np.array(py["embedding"].tolist(), dtype="float32")
py[["x", "y"]] = PCA(n_components=2, random_state=42).fit_transform(X)

# drop the non-party outlier; drop very noisy low-count party-years
py = py[py["party"] != "Fraktionslos"].copy()
if "n_speeches" in py.columns:
    py = py[py["n_speeches"] >= 20].copy()

**Setup.** We reduce the 1024-dim party-year embeddings to 2D with PCA, then clean the data: drop *Fraktionslos* (not a real party — a noisy grab-bag of independents that otherwise dominates the plot) and drop any party-year built from fewer than 20 speeches (its average is too noisy to trust).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for party, g in py.sort_values("year").groupby("party"):
    c = party_colors.get(party, "grey")
    x, y = g["x"].values, g["y"].values
    ax.quiver(x[:-1], y[:-1], np.diff(x), np.diff(y),          # year -> year arrows
              angles="xy", scale_units="xy", scale=1, color=c, width=0.003, alpha=0.7)
    ax.scatter(x[0],  y[0],  facecolors="none", edgecolors=c, s=30)   # start (open)
    ax.scatter(x[-1], y[-1], color=c, s=45, label=party)             # end (filled)
ax.legend(); ax.axis("off")
ax.set_title("Party trajectories in embedding space (arrows = direction over time)")
plt.tight_layout(); plt.show()

**Party trajectories in embedding space.** Each arrow is one party's move from one year to the next; the open circle is its first year, the filled dot its last. Two things stand out. The paths are **tangled together** rather than running in separate lanes, and every party's **filled endpoint sits in the same corner** of the plot — they end up closer to each other than they started. Note these are PC1/PC2 (the *dominant* axes of variation), which mostly capture topic and era, so read this as *how much* a party's average speech shifted over time, not *in which political direction*. The two long, straight arrows (FDP, AfD) are not big moves either: they bridge years in which the party was absent from parliament.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for party, g in py.sort_values("year").groupby("party"):
    ax.plot(g["year"], g["x"], marker="o", color=party_colors.get(party, "grey"), label=party)
# mark known events
for yr in [2005, 2009, 2013, 2015, 2017, 2020, 2021]:
    ax.axvline(yr, color="grey", ls="--", alpha=0.3)
ax.set_xlabel("year"); ax.set_ylabel("position (PC1 of embedding)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("Party position over time")
plt.tight_layout(); plt.show()

**Party position over time (PC1).** Each line is a party's position on the first principal component across the years; dashed lines mark elections/crises. The parties are **bunched** — in most years they sit within about 0.03 of each other — and the whole set **drifts upward together**, from roughly −0.08 in 2000 to about +0.11 in 2021, with the steepest part after 2013. A shared, parallel move like that is a **time / era effect**: PC1 is capturing how parliamentary language changed over the period, not left–right movement. The FDP's straight diagonal from 2013 to 2017 is a gap, not a jump — the party was out of parliament and has no party-years in between.

In [ ]:
E = np.array(py["embedding"].tolist(), dtype="float32")      # (n_party_years, dim)

# left–right axis = (mean of right-anchor party-years) - (mean of left-anchor party-years)
left_anchor  = ["LINKE", "PDS"]        # far-left pole (present most of the period)
right_anchor = ["CDU/CSU"]             # mainstream-right pole; swap in "AfD" for a harder-right axis (2017+ only)

left_mean  = E[py["party"].isin(left_anchor).to_numpy()].mean(axis=0)
right_mean = E[py["party"].isin(right_anchor).to_numpy()].mean(axis=0)
axis = right_mean - left_mean
axis = axis / np.linalg.norm(axis)

py["lr"] = E @ axis                                          # higher = closer to the right pole
py["lr"] = (py["lr"] - py["lr"].mean()) / py["lr"].std()     # standardise for readability

# sanity check: parties should order roughly left -> right
print(py.groupby("party")["lr"].mean().sort_values())

events = [2005, 2009, 2013, 2015, 2017, 2020, 2021]

**Does the axis capture left–right? Only partly.** Ignore LINKE (−1.89) and CDU/CSU (+0.84) — they're the anchors, so they sit at the extremes by construction. Grüne (−0.06), FDP (+0.21) and PDS (−0.57) land sensibly, but **AfD (−0.80) and SPD (+0.75) are badly misplaced**: the AfD should be far right, the SPD centre-left, and here the SPD sits closer to the Union than any other party does. This is the classic *horseshoe* — AfD's oppositional rhetoric resembles LINKE's (both outsiders), and long-governing SPD resembles CDU/CSU (both establishment). So the embedding axis is really tracking **government/opposition and topic, not ideology**. Important caveat for any 'who moved left/right' reading — and a concrete example of where embeddings fall short of purpose-built scaling methods like Wordfish/Wordscores.

## 2. Analyze Trajectories

We test if the following political events appear as detectable movements in the embedding space
(embeddings aggregated by MP and year):

- 9/11 terror attacks
- Lehman Brothers collapse / global financial crisis
- Fukushima nuclear disaster
- Shift to migration scepticism in Germany (post-Cologne NYE)

To test, whether those events trigger movements of the embeddings, we first have to filter for
related topics. The four `TOPICS_*` lists defined below are the single topic–event assignment used
by the rest of the notebook — part 4 builds its `EVENTS` list from exactly these lists.

In [ ]:
# List of all topics
print(df_embeddings_topics.topic_emb_full_name.unique())

### 9/11 terror attacks
Zero-shot category related to the 9/11 terror attacks:
- 'Verteidigungspolitik und Bundeswehr' ("defence policy / Bundeswehr") — the corpus records the
  parliamentary consequences, the deployment mandates, rather than the attack itself.


In [ ]:
TOPICS_9_11 = [
    'Verteidigungspolitik und Bundeswehr'
]

df_9_11 = df_embeddings_topics[df_embeddings_topics.topic_emb_raw_name.isin(TOPICS_9_11)]

df_9_11

### Lehman Brothers collapse / global financial crisis
Zero-shot category related to the Lehman Brothers collapse / global financial crisis:
- 'Steuern und Finanzpolitik' ("taxes and financial policy")


In [ ]:
TOPICS_FINANCE = [
    'Steuern und Finanzpolitik'
]

df_financial_crisis = df_embeddings_topics[df_embeddings_topics.topic_emb_full_name.isin(TOPICS_FINANCE)]
df_financial_crisis

### Fukushima nuclear disaster
Zero-shot category related to the Fukushima nuclear disaster:
- 'Umweltschutz, Klimapolitik und Atomkraft' ("environmental protection, climate policy and nuclear power")


In [ ]:
TOPICS_FUKUSHIMA = [
    'Umweltschutz, Klimapolitik und Atomkraft'
]

df_fukushima = df_embeddings_topics[df_embeddings_topics.topic_emb_full_name.isin(TOPICS_FUKUSHIMA)]
df_fukushima

### Shift to migration scepticism in Germany (post-Cologne NYE)
Zero-shot category related to the shift to migration scepticism (post-Cologne NYE):
- 'Migration, Asyl und Einwanderung' ("migration, asylum and immigration")


In [ ]:
TOPICS_MIGRATION = [
    'Migration, Asyl und Einwanderung'
]

df_migration = df_embeddings_topics[df_embeddings_topics.topic_emb_raw_name.isin(TOPICS_MIGRATION)]

df_migration

### Plot trajectories

In [ ]:
def plot_trajectory(df, year, timespan, topic):

    # restrict to the years around the event (to make plots more readable)
    df = df[(df["year"] >= timespan[0]) & (df["year"] <= timespan[1])]

    # aggregate the filtered embeddings by party and year
    df_agg = aggregate_by_party_year(df)

    # stack embeddings to array
    stack = np.stack(df_agg['embedding'])

    # PCA to reduce embedding space to two dimensions
    pca_traj = PCA(n_components=2)
    pcs_traj = pca_traj.fit_transform(stack)

    # data frame for plot
    df_party_trajectories = pd.DataFrame({
        "dim1": pcs_traj[:, 0],
        "dim2": pcs_traj[:, 1],
        "party": df_agg.party,
        "year": df_agg.year
    })

    df_party_trajectories = df_party_trajectories.sort_values(["party", "year"])

    # plot trajectories
    for party, color in party_colors.items():
        subset = df_party_trajectories[df_party_trajectories["party"] == party]
        if len(subset) == 0:
            continue

        plt.plot(subset["dim1"], subset["dim2"], color=color, label=party, marker='o', markersize=3)

        if len(subset) >= 2:
            plt.annotate('', xy=(subset["dim1"].iloc[1], subset["dim2"].iloc[1]),
                                xytext=(subset["dim1"].iloc[0], subset["dim2"].iloc[0]),
                                arrowprops=dict(arrowstyle='->', color=color, lw=2))

        # label the point for the given event year, if present
        year_row = subset[subset["year"] == year]
        if not year_row.empty:
            plt.annotate(str(year),
                        (year_row["dim1"].iloc[0], year_row["dim2"].iloc[0]),
                        fontsize=8, color=color,
                        textcoords="offset points", xytext=(5, 5))

    plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.axis("off")
    plt.title(f"Trajectories of Party-Year Embeddings ({topic})")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_trajectory(df_9_11, 2001, [2000, 2010], 'Terrorism')

In [ ]:
plot_trajectory(df_financial_crisis, 2008, [2005, 2015], 'Financial Politics')

In [ ]:
plot_trajectory(df_fukushima, 2011, [2005, 2015], 'Nuclear Energy')

In [ ]:
plot_trajectory(df_migration, 2016, [2011, 2021], 'Migration')

In [ ]:
def plot_trajectory_per_party(df, year, topic):

    # aggregate the embeddings by party and year (no time restriction)
    df_agg = aggregate_by_party_year(df)

    # stack embeddings to array
    stack = np.stack(df_agg['embedding'])

    # PCA to reduce embedding space to two dimensions
    pca_traj = PCA(n_components=2)
    pcs_traj = pca_traj.fit_transform(stack)

    # data frame for plot
    df_party_trajectories = pd.DataFrame({
        "dim1": pcs_traj[:, 0],
        "dim2": pcs_traj[:, 1],
        "party": df_agg.party,
        "year": df_agg.year
    })

    df_party_trajectories = df_party_trajectories.sort_values(["party", "year"])

    # one plot per party
    for party, color in party_colors.items():
        subset = df_party_trajectories[df_party_trajectories["party"] == party]
        if len(subset) == 0:
            continue

        plt.figure(figsize=(6, 5))
        plt.plot(subset["dim1"], subset["dim2"], color=color, marker='o', markersize=3)

        # arrows between all consecutive points, to show the full trajectory direction
        for i in range(len(subset) - 1):
            plt.annotate('', xy=(subset["dim1"].iloc[i+1], subset["dim2"].iloc[i+1]),
                                xytext=(subset["dim1"].iloc[i], subset["dim2"].iloc[i]),
                                arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.7))

        # label every year
        for i in range(len(subset)):
            plt.annotate(str(subset["year"].iloc[i]),
                        (subset["dim1"].iloc[i], subset["dim2"].iloc[i]),
                        fontsize=8, color=color,
                        textcoords="offset points", xytext=(5, 5))

        # highlight the event year, if present
        year_row = subset[subset["year"] == year]
        if not year_row.empty:
            plt.scatter(year_row["dim1"], year_row["dim2"], color=color,
                        s=80, edgecolors="black", zorder=5)

        plt.axis("off")
        plt.title(f"{party} — Trajectory of Year Embeddings ({topic})")
        plt.tight_layout()
        plt.show()

In [ ]:
plot_trajectory_per_party(df_9_11, 2001, 'Terrorism')

In [ ]:
plot_trajectory_per_party(df_financial_crisis, 2008, 'Financial Politics')

In [ ]:
plot_trajectory_per_party(df_fukushima, 2011, 'Nuclear Energy')

In [ ]:
plot_trajectory_per_party(df_migration, 2016, 'Migration')

In [ ]:
def plot_trajectory_1d(df, year, topic):

    # aggregate the embeddings by party and year
    df_agg = aggregate_by_party_year(df)

    # stack embeddings to array
    stack = np.stack(df_agg['embedding'])

    # PCA to reduce embedding space to one dimension
    pca_traj = PCA(n_components=1)
    pcs_traj = pca_traj.fit_transform(stack)

    # data frame for plot
    df_party_trajectories = pd.DataFrame({
        "pc1": pcs_traj[:, 0],
        "party": df_agg.party,
        "year": df_agg.year
    })

    df_party_trajectories = df_party_trajectories.sort_values(["party", "year"])

    for party, color in party_colors.items():
        subset = df_party_trajectories[df_party_trajectories["party"] == party]
        if len(subset) == 0:
            continue
        plt.plot(subset["year"], subset["pc1"], color=color, label=party, marker='o', markersize=3)

    # mark the event year
    plt.axvline(year, color="grey", linestyle="--", linewidth=1)

    plt.xlabel("Year")
    plt.ylabel("PC1")
    plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.title(f"PC1 of Party-Year Embeddings over Time ({topic})")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_trajectory_1d(df_9_11, 2001, 'Terrorism')

In [ ]:
plot_trajectory_1d(df_financial_crisis, 2008, 'Financial Politics')

In [ ]:
plot_trajectory_1d(df_fukushima, 2011, 'Nuclear Energy')

In [ ]:
plot_trajectory_1d(df_migration, 2016, 'Migration')

In [ ]:
def demean_by_year(df_traj, value_col="pc1"):
    """Subtracts the across-party mean per year from the given value column."""
    df_traj = df_traj.copy()
    year_means = df_traj.groupby("year")[value_col].transform("mean")
    df_traj[f"{value_col}_demeaned"] = df_traj[value_col] - year_means
    return df_traj

In [ ]:
def plot_trajectory_1d_demeaned(df, year, topic):

    # aggregate the embeddings by party and year
    df_agg = aggregate_by_party_year(df)

    # stack embeddings to array
    stack = np.stack(df_agg['embedding'])

    # PCA to reduce embedding space to one dimension
    pca_traj = PCA(n_components=1)
    pcs_traj = pca_traj.fit_transform(stack)

    # data frame for plot
    df_party_trajectories = pd.DataFrame({
        "pc1": pcs_traj[:, 0],
        "party": df_agg.party,
        "year": df_agg.year
    })

    df_party_trajectories = df_party_trajectories.sort_values(["party", "year"])

    # subtract the across-party mean per year (to remove common, year-wide shifts)
    df_party_trajectories = demean_by_year(df_party_trajectories, value_col="pc1")

    for party, color in party_colors.items():
        subset = df_party_trajectories[df_party_trajectories["party"] == party]
        if len(subset) == 0:
            continue
        plt.plot(subset["year"], subset["pc1_demeaned"], color=color, label=party, marker='o', markersize=3)

    plt.axvline(year, color="grey", linestyle="--", linewidth=1)
    plt.xlabel("Year")
    plt.ylabel("PC1 (demeaned per year)")
    plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.title(f"PC1 of Party-Year Embeddings over Time, Demeaned ({topic})")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_trajectory_1d_demeaned(df_9_11, 2001, "Terrorism")

In [ ]:
plot_trajectory_1d_demeaned(df_financial_crisis, 2008, "Financial Politics")

In [ ]:
plot_trajectory_1d_demeaned(df_fukushima, 2011, "Nuclear Energy")

In [ ]:
plot_trajectory_1d_demeaned(df_migration, 2015, "Migration")

We therefore try two alternative approaches to the visual inspection of party trajectories:
- instead of tracking whole parties, we focus on individual politicians who specialized in the respective topic
- instead of relying on a visual check, we compute the cosine similarity between consecutive embeddings and look for peaks in the resulting distance, indicating unusually large shifts.


In [ ]:
def find_specialist_per_party(data, topic_name):
    """
    Für ein gegebenes Topic: pro Partei den Politiker/die Politikerin mit den
    meisten Reden zu diesem Topic auswählen.
    """
    topic_speeches = data[data["topic_emb_raw_name"] == topic_name]

    specialists = {}
    for party, group in topic_speeches.groupby("party"):
        top_politician = group["politicianId"].value_counts().idxmax()
        specialists[party] = top_politician

    return specialists

In [ ]:
# the first (main) topic of each event defines its specialists
specialists_9_11 = find_specialist_per_party(df_embeddings_topics, TOPICS_9_11[0])
specialists_finance = find_specialist_per_party(df_embeddings_topics, TOPICS_FINANCE[0])
specialists_nuclear_energy = find_specialist_per_party(df_embeddings_topics, TOPICS_FUKUSHIMA[0])
specialists_migration = find_specialist_per_party(df_embeddings_topics, TOPICS_MIGRATION[0])

In [ ]:
def get_specialist_trajectories(data, specialists):
    """specialists: Dict {party: politicianId} aus find_specialist_per_party."""
    politician_ids = list(specialists.values())
    df_specialists = data[data["politicianId"].isin(politician_ids)]
    return aggregate_by_mp_year(df_specialists)

In [ ]:
df_specialists_9_11 = get_specialist_trajectories(df_9_11, specialists_9_11)
df_specialists_9_11

In [ ]:
df_specialists_finance = get_specialist_trajectories(df_financial_crisis, specialists_finance)
df_specialists_finance

In [ ]:
df_specialists_fukushima = get_specialist_trajectories(df_fukushima, specialists_nuclear_energy)
df_specialists_fukushima

In [ ]:
df_specialists_migration = get_specialist_trajectories(df_migration, specialists_migration)
df_specialists_migration

In [ ]:
def plot_trajectory_1d_specialists(df, year, topic, specialists):
    df_agg = aggregate_by_mp_year(df)

    stack = np.stack(df_agg['embedding'])
    pca_traj = PCA(n_components=1)
    pcs_traj = pca_traj.fit_transform(stack)

    df_traj = pd.DataFrame({
        "pc1": pcs_traj[:, 0],
        "politicianId": df_agg.politicianId,
        "party": df_agg.party,
        "year": df_agg.year
    })
    df_traj = df_traj.sort_values(["politicianId", "year"])

    for party, politician_id in specialists.items():
        subset = df_traj[df_traj["politicianId"] == politician_id]
        if len(subset) == 0:
            continue
        plt.plot(subset["year"], subset["pc1"], color=party_colors[party],
                  label=f"{party} ({politician_id})", marker='o', markersize=3)

    plt.axvline(year, color="grey", linestyle="--", linewidth=1)
    plt.xlabel("Year")
    plt.ylabel("PC1")
    plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.title(f"PC1 of Specialist MP Embeddings over Time ({topic})")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_trajectory_1d_specialists(df_specialists_9_11, 2001, 'Terrorism', specialists_9_11)

In [ ]:
plot_trajectory_1d_specialists(df_specialists_finance, 2008, 'Financial Politics', specialists_finance)

In [ ]:
plot_trajectory_1d_specialists(df_specialists_fukushima, 2011, 'Nuclear Energy', specialists_nuclear_energy)

In [ ]:
plot_trajectory_1d_specialists(df_specialists_migration, 2015, 'Migration', specialists_migration)

### Cosine Similarity

In [ ]:
def compute_distance_series(df, topic):
    """
    Aggregates the (already topic-filtered) speeches by party and year,
    then computes the cosine distance between each party's embedding
    in consecutive years.
    """
    df_agg = aggregate_by_party_year(df)
    df_agg = df_agg.sort_values(["party", "year"])

    records = []
    for party, color in party_colors.items():
        subset = df_agg[df_agg["party"] == party].sort_values("year")
        if len(subset) < 2:
            continue

        embeddings = np.stack(subset["embedding"])
        years = subset["year"].values

        for i in range(1, len(embeddings)):
            sim = cosine_similarity(embeddings[i-1].reshape(1, -1), embeddings[i].reshape(1, -1))[0, 0]
            distance = 1 - sim
            records.append({
                "party": party,
                "year": years[i],
                "distance": distance
            })

    return pd.DataFrame(records)

def plot_distance_series(df, topic, event_year):
    df_dist = compute_distance_series(df, topic)

    for party, color in party_colors.items():
        subset = df_dist[df_dist["party"] == party]
        if len(subset) == 0:
            continue
        plt.plot(subset["year"], subset["distance"], color=color, label=party, marker='o', markersize=3)

    plt.axvline(event_year, color="grey", linestyle="--", linewidth=1)
    plt.xlabel("Year")
    plt.ylabel("Cosine Distance to Previous Year")
    plt.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.title(f"Year-over-Year Embedding Distance ({topic})")
    plt.xticks(sorted(df_dist["year"].unique()), rotation=45)
    plt.tight_layout()
    plt.show()

    return df_dist

In [ ]:
df_dist_9_11 = plot_distance_series(df_9_11, "Terrorism", 2001)
df_dist_9_11

In [ ]:
df_dist_finance = plot_distance_series(df_financial_crisis, "Financial Politics", 2008)
df_dist_finance

In [ ]:
df_dist_fukushima = plot_distance_series(df_fukushima, "Nuclear Energy", 2011)
df_dist_fukushima

In [ ]:
df_dist_migration = plot_distance_series(df_migration, "Migration", 2015)
df_dist_migration

## 3. Party anchors — positions on an anchored axis

Part 2 asked where PCA puts the parties, and found that the dominant direction tracks era and
government/opposition rather than policy. This part replaces that with an axis we choose: two
parties are named as poles, and everyone else is projected onto the line between them.
Sample sizes behind these points are shown by marker size.

- yearly mean is subtracted so a drift shared by all parties cannot look like movement;
- the axis is estimated on the years before the event so the ruler does not move with the data; and the result is rescaled so the two anchor parties sit at 0 and 1 in that reference period, which makes a value of 0.4 readable as "40% of the way from one party's position to the other's"

In [ ]:
def positions(topic_names, left, right, ref_years):
    "Party-year positions on the left->right axis, rescaled so left=0 and right=1 before the event."
    d = df[df.topic_emb_full_name.isin(topic_names)]

    # 1. aggregate to party-year, no threshold
    py = pd.DataFrame([dict(party=p, year=y, n=len(g),
                            E=np.mean(np.stack(g.E.to_numpy()), axis=0))
                       for (p, y), g in d.groupby(["party", "year"])])

    # a year with one party is useless: the yearly mean is that party, so demeaning gives 0
    per_year = py.groupby("year").size()
    lonely = per_year[per_year < 2].index.tolist()
    if lonely:
        print(f"years with only one party, dropped: {lonely}")
        py = py[~py.year.isin(lonely)]

    # 2. subtract the yearly mean
    ymu = {y: np.mean(np.stack(g.E.to_numpy()), axis=0) for y, g in py.groupby("year")}
    py["Ed"] = [e - ymu[y] for e, y in zip(py.E, py.year)]

    # 3. axis from the reference (pre-event) years only
    ref = py[py.year.isin(ref_years)]
    for anchor in (left, right):
        if not (ref.party == anchor).any():
            raise ValueError(f"{anchor} has no speech in the reference years {list(ref_years)}")
    a = (np.mean(np.stack(ref.loc[ref.party == right, "Ed"].to_numpy()), axis=0)
         - np.mean(np.stack(ref.loc[ref.party == left,  "Ed"].to_numpy()), axis=0))
    axis = a / np.linalg.norm(a)

    # 4. project, then rescale so the anchors sit at 0 and 1 in the reference period
    py["raw"] = [float(e @ axis) for e in py.Ed]
    lo = py.loc[py.year.isin(ref_years) & (py.party == left),  "raw"].mean()
    hi = py.loc[py.year.isin(ref_years) & (py.party == right), "raw"].mean()
    py["pos"] = (py.raw - lo) / (hi - lo)
    return py.sort_values(["party", "year"])


def plot_positions(py, event_year, left, right, title):
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    ax.axhline(0, color=PARTY_COLORS.get(left, "0.4"), ls=":", lw=1.4)
    ax.axhline(1, color=PARTY_COLORS.get(right, "0.4"), ls=":", lw=1.4)
    ax.axvline(event_year, color="#c0392b", ls="--", lw=1.5)
    for p, g in py.groupby("party"):
        g = g.sort_values("year")
        col = PARTY_COLORS.get(p, "grey")
        # break the line where a year is missing, so a gap is not drawn as a smooth move
        run = []
        for r in g.itertuples():
            if run and r.year - run[-1][0] > 1:
                ax.plot(*zip(*run), "-", color=col, lw=1.8)
                run = []
            run.append((r.year, r.pos))
        if len(run) > 1:
            ax.plot(*zip(*run), "-", color=col, lw=1.8)
        # marker size shows how many speeches are behind the point
        ax.scatter(g.year, g.pos, s=6 + 1.6 * g.n, color=col, zorder=4, label=p,
                   edgecolor="white", linewidth=.6)
    ax.set_xlabel("year")
    ax.set_ylabel(f"position on the {left} → {right} axis\n(0 = {left}, 1 = {right} before the event)")
    ax.set_title(title + "   (marker size = speeches behind the point)", loc="left", fontsize=10)
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    return ax

#### Fukushima — did the Union move toward the Greens?

The clearest test case, because the answer is documented outside the corpus: after March 2011
the CDU/CSU reversed course and legislated the nuclear phase-out it had extended only months
earlier. If the embeddings carry policy content, the Union should move upward after 2011. The
axis is estimated on 2005–2010.

In [ ]:
NUCLEAR = TOPICS_FUKUSHIMA

py_nuc = positions(NUCLEAR, left="CDU/CSU", right="Grüne", ref_years=range(2005, 2011))
print("speeches per party-year:")
print(py_nuc.pivot(index="year", columns="party", values="n").fillna(0).astype(int).to_string())

plot_positions(py_nuc, 2011, "CDU/CSU", "Grüne", "Nuclear topics — CDU/CSU vs Grüne")
plt.savefig("fig_fukushima.png", dpi=150, bbox_inches="tight"); plt.show()

before = py_nuc[py_nuc.year.between(2007, 2010)].groupby("party").pos.mean()
after  = py_nuc[py_nuc.year.between(2012, 2015)].groupby("party").pos.mean()
print(pd.DataFrame({"2007-2010": before, "2012-2015": after,
                    "shift": after - before}).round(2).to_string())

#### Migration — where do the parties sit relative to the AfD?

The AfD only enters the Bundestag in 2017, so the axis has to be estimated *after* the event
rather than before it. That weakens the reading considerably: we can see the ordering of the
parties from 2017 onward, but not how they stood relative to a party that did not yet exist.

Anchors: Grüne und AfD

In [ ]:
MIGRATION = TOPICS_MIGRATION

py_mig = positions(MIGRATION, left="Grüne", right="AfD", ref_years=range(2018, 2022))
print("speeches per party-year:")
print(py_mig.pivot(index="year", columns="party", values="n").fillna(0).astype(int).to_string())

plot_positions(py_mig, 2016, "Grüne", "AfD", "Migration topic — Grüne vs AfD")
plt.savefig("fig_migration.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Topic frequencies

Does an event change how much is said about a topic at all? This part never touches the embedding
vectors — only the topic labels and the speech counts behind them.

In [ ]:
EVENTS = [
    #Afghanistan topics stand in for 9/11, since corpus records the parliamentary consequences rather than the attack
    dict(date="2001-09-11", label="9/11 terror attacks", kind="crisis", topics=TOPICS_9_11),
    dict(date="2008-09-15", label="Lehman collapse / financial crisis", kind="crisis", topics=TOPICS_FINANCE),
    dict(date="2011-03-11", label="Fukushima nuclear disaster", kind="crisis", topics=TOPICS_FUKUSHIMA),
    dict(date="2016-01-01", label="Migration scepticism (post-Cologne NYE)", kind="crisis", topics=TOPICS_MIGRATION),
]

for ev in EVENTS:
    ev["year"] = int(ev["date"][:4])
#checks that every name still exists in the data, so a re-run of the topic model would raise an error rather than silently analysing nothing.
unknown = {t for ev in EVENTS for t in ev["topics"]} - set(ALL_TOPICS)
if unknown:
    raise ValueError(f"topic name(s) not found in the data: {unknown}")
print("all topic names found in the data\n")

for ev in EVENTS:
    n = df.topic_emb_full_name.isin(ev["topics"]).sum()
    print(f"{ev['date']}  {ev['label']}")
    for t in ev["topics"]:
        print(f"     - {t}")
    print(f"     {n:,} speeches in total\n")

### Does attention change around the event?

Two views per event.

**Attention over time.** The share of all speeches in a year that fall on the
event's topics. If a crisis reorganises the agenda, this is where it shows up most directly.

**Who is doing the talking.** For each party, the share of *its own* speeches
devoted to those topics, in the three years before and the three years after the event. Using
each party's own denominator means a party that simply speaks more often does not appear more
interested.

**Share of each party's speeches on topic**

In [ ]:
WINDOW = 3      # years before / after the event used for the party comparison

def frequency_figure(ev):
    "Attention to an event's topics: over time, and per party before vs after."
    on_topic = df.topic_emb_full_name.isin(ev["topics"])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8),
                                   gridspec_kw={"width_ratios": [1.5, 1]})

    # --- left: share of the year's speeches, one line per topic plus the total
    for t in ev["topics"]:
        share = df.topic_emb_full_name.eq(t).groupby(df.year).mean()
        ax1.plot(share.index, 100 * share.values, lw=1.4, alpha=.75,
                 label=t.split("/")[0][:22])
    if len(ev["topics"]) > 1:
        tot = on_topic.groupby(df.year).mean()
        ax1.plot(tot.index, 100 * tot.values, lw=2.4, color="0.2", label="all topics together")
    ax1.axvline(ev["year"], color="#c0392b", ls="--", lw=1.5)
    ax1.set_xlabel("year"); ax1.set_ylabel("% of that year's speeches")
    ax1.set_title(f"{ev['label']} ({ev['year']})", loc="left", fontsize=10)
    ax1.legend(fontsize=7)

    # --- right: per party, share of its own speeches, before vs after
    pre  = df[df.year.between(ev["year"] - WINDOW, ev["year"] - 1)]
    post = df[df.year.between(ev["year"] + 1, ev["year"] + WINDOW)]
    parties = [p for p in PARTY_COLORS if p in set(pre.party) | set(post.party)]
    b = [100 * pre.loc[pre.party == p, "topic_emb_full_name"].isin(ev["topics"]).mean()
         for p in parties]
    a = [100 * post.loc[post.party == p, "topic_emb_full_name"].isin(ev["topics"]).mean()
         for p in parties]
    x = np.arange(len(parties))
    ax2.bar(x - 0.2, b, 0.4, color="0.75", label=f"{WINDOW}y before")
    ax2.bar(x + 0.2, a, 0.4, color=[PARTY_COLORS[p] for p in parties], label=f"{WINDOW}y after")
    ax2.set_xticks(x); ax2.set_xticklabels(parties, rotation=30, ha="right", fontsize=8)
    ax2.set_ylabel("% of the party's speeches")
    ax2.set_title("attention per party", loc="left", fontsize=10)
    ax2.legend(fontsize=7)

    plt.tight_layout()
    slug = "".join(ch if ch.isalnum() else "_" for ch in ev["label"].lower())[:28]
    plt.savefig(f"fig_freq_{ev['year']}_{slug}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # --- the same numbers as a table: party x topic, share of the party's speeches
    d = df[on_topic]
    tab = (100 * pd.crosstab(d.party, d.topic_emb_full_name).div(
           df.groupby("party").size(), axis=0)).round(2)
    tab.columns = [c.split("/")[0][:22] for c in tab.columns]
    print(f"share of each party's speeches on these topics, % — {ev['label']}")
    print(tab.to_string(), "\n")
    return tab


for ev in EVENTS:
    frequency_figure(ev)